# Imports

In [52]:
from src.get_gender_data import get_founder_gender
from src.scrape_wiki import scrape_wiki_table
from src.get_gender_data import get_founder_gender, _parse_names
from src.filter_df import filter_df

import pandas as pd
import os

# Functions

In [53]:
def rescue_unknowns(df, founder_col, gender_col):
    """Re-process only rows where gender is unknown or contains unknown."""
    mask = df[gender_col].apply(lambda x: 'unknown' in str(x).lower() if x is not None else True)
    unknown_count = mask.sum()

    print(f"Attempting to rescue {unknown_count} unknowns in {gender_col}...")

    # Targeted update
    df.loc[mask, gender_col] = df.loc[mask, founder_col].apply(lambda x: get_founder_gender(x, use_web_search=True))

    new_unknown_count = df[gender_col].apply(lambda x: 'unknown' in str(x).lower()).sum()
    print(f"Resolution complete. Unknowns remaining: {new_unknown_count} (Rescued {unknown_count - new_unknown_count})")
    return df

In [54]:


# Ensure output directory exists
os.makedirs("processed_data", exist_ok=True)

def all_unknown_genders(founder_string):
    """Skip gender prediction entirely and return 'unknown' for every founder."""
    if founder_string is None or (isinstance(founder_string, float) and pd.isna(founder_string)):
        return []
    return ['unknown'] * len(_parse_names(str(founder_string).strip()))

In [55]:
url = "https://en.wikipedia.org/wiki/List_of_unicorn_startup_companies"
idx = 2  # Changed to target the main list of unicorn companies

currentUnicorns_df = scrape_wiki_table(url, idx, "current_unicorns.csv")
currentUnicorns_df.to_csv("current_unicorns.csv")

pastUnicorns_df = scrape_wiki_table(url, idx+1, "past_unicorns.csv")
pastUnicorns_df.to_csv("past_unicorns.csv")

Found 4 wikitable(s)
Saved 619 rows to current_unicorns.csv
Found 4 wikitable(s)
Saved 207 rows to past_unicorns.csv


In [56]:
currentUnicorns_df = pd.read_csv("current_unicorns.csv", index_col=0)
pastUnicorns_df = pd.read_csv("past_unicorns.csv", index_col=0)



In [57]:
print("Predicting genders for current unicorns...")

currentUnicorns_df['Founder_Genders'] = filter_df(currentUnicorns_df, "Founder(s)", all_unknown_genders)

# Save intermediate result
currentUnicorns_df.to_csv("processed_data/current_unicorns_with_gender.csv", index=False)

df_companies_founders_gender = currentUnicorns_df[['Company', 'Founder_Genders', 'Industry']].copy()
# Note: get_founder_gender now returns normalized 'male', 'female', 'unknown'
print("Done.")

Predicting genders for current unicorns...
Done.


In [58]:
display(currentUnicorns_df)
display(pastUnicorns_df)

,Company,Valuation(US$ billions),Valuation date,Industry,Country/countries,Founder(s),Founder_Genders
0,Anthropic,965,May 2026(2026-05)[20],Artificial Intelligence,United States,"Dario Amodei,Daniela Amodei,Jared Kaplan, Jack...","[unknown, unknown, unknown, unknown, unknown, ..."
1,OpenAI,852,March 2026(2026-03)[21],Artificial intelligence,United States,"Sam Altman,Elon Musk,Greg Brockman,Ilya Sutskever","[unknown, unknown, unknown, unknown]"
2,ByteDance,600,April 2026[22],Internet,China,"Zhang Yiming, Liang Rubo","[unknown, unknown]"
3,Stripe,159,February 2026(2026-02)[23],Financial services,United States and Ireland,PatrickandJohn Collison,"[unknown, unknown]"
4,Databricks,134,December 2025(2025-12)[24],Software,United States,"Ali Ghodsi,Andy Konwinski,Ion Stoica,Reynold X...","[unknown, unknown, unknown, unknown, unknown]"
...,...,...,...,...,...,...,...
614,HMD Global,1+,August 2020[570],Mobile Devices,Finland,Jean-Francois Baril,[unknown]
615,IQM,1,July 2022[571],Quantum Computing,Finland,"Jan Goetz, Mikko Möttönen, Kuan Yen Tan, Juha ...","[unknown, unknown, unknown, unknown]"
616,Hostaway,1,December 2024[572],Vacation rental software platform,Finland,"Marcus Räder, Saber Kordestanchi, Mikko Nurminen","[unknown, unknown, unknown]"
617,Papara,1,July 2023[573],Fintech,Turkey,Ahmed Faruk Karslı,[unknown]


,Company,Last valuation(US$billions),Valuation date,Exit date,Exit reason,Exit valuation(US$billions),Country,Founders,col_8
0,SpaceX,1250,February 2026(2026-02)[576],June 2026(2026-06)[577],IPO,1770,United States,Elon Musk,NaN
1,Uber,72,August 2018[578],May 2019[579],IPO,82.4,United States,"Travis Kalanick,Garett Camp",NaN
2,DiDi,62,July 2019[580],June 2021[581],IPO,73,China,Cheng Wei,NaN
3,Facebook,50,January 2011,May 2012[582],IPO,104,United States,"Mark Zuckerberg,Eduardo Saverin, Andrew McColl...",NaN
4,Xiaomi,45,April 2015,July 2018[583],IPO,70,China,Lei Jun,NaN
...,...,...,...,...,...,...,...,...,...
202,Zimi,1+,February 2015[184],March 2021[837],Acquired,0.4,China,NaN,NaN
203,QingCloud,1+,June 2017[184],March 2021[838],IPO,0.46,China,NaN,NaN
204,Novogene,1+,November 2016[184],April 2021,IPO,NaN,China,NaN,NaN
205,MissFresh,1+,December 2017[184],June 2021[839],IPO,2.5,China,NaN,NaN


will run for several minutes:

In [59]:
currentUnicorns_df['Founder_Genders'] = filter_df(currentUnicorns_df, "Founder(s)", all_unknown_genders)


In [60]:

print("Processing Current Unicorns...")
currentUnicorns_df = rescue_unknowns(currentUnicorns_df, 'Founder(s)', 'Founder_Genders')

print("\nProcessing Past Unicorns...")
pastUnicorns_df = rescue_unknowns(pastUnicorns_df, 'Founders', 'Founder_Genders')

# Save the improved data
currentUnicorns_df.to_csv('processed_data/current_unicorns.csv', index=False)
pastUnicorns_df.to_csv('processed_data/past_unicorns.csv', index=False)

Processing Current Unicorns...
Attempting to rescue 208 unknowns in Founder_Genders...
Resolution complete. Unknowns remaining: 0 (Rescued 208)

Processing Past Unicorns...


KeyError: 'Founder_Genders'